In [0]:
%pip install sentence-transformers

In [0]:
dbutils.library.restartPython()

In [0]:
import base64
from urllib.parse import urlparse
from databricks.sdk import WorkspaceClient
import psycopg2

w = WorkspaceClient()
secret = w.secrets.get_secret(scope="database", key="lakebase-url")
connection_string = base64.b64decode(secret.value).decode("utf-8")
parsed = urlparse(connection_string)

conn = psycopg2.connect(
    host=parsed.hostname, port=parsed.port or 5432, dbname=parsed.path.lstrip("/"),
    user=parsed.username, password=parsed.password, sslmode="require", connect_timeout=10,
)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS destination_description_embeddings (
        id SERIAL PRIMARY KEY,
        destination TEXT,
        title TEXT,
        summary TEXT,
        wikipedia_url TEXT,
        feature TEXT,
        embedding_text TEXT,
        embedding VECTOR(384),
        embedded_at TIMESTAMP DEFAULT NOW()
    );
""")
conn.commit()
cursor.close()
conn.close()
print("Table ready.")

In [0]:
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')  # Suppress HF token warnings

silver_descriptions = spark.table("trip_planner.silver.destination_descriptions")
pdf = silver_descriptions.toPandas()

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(pdf["embedding_text"].tolist())
pdf["embedding"] = embeddings.tolist()

conn = psycopg2.connect(
    host=parsed.hostname, port=parsed.port or 5432, dbname=parsed.path.lstrip("/"),
    user=parsed.username, password=parsed.password, sslmode="require",
)
cursor = conn.cursor()

# Use only the columns that exist in the original table schema
insert_sql = """
    INSERT INTO destination_description_embeddings
        (destination, title, summary, embedding)
    VALUES (%s, %s, %s, %s::vector)
"""

for _, row in pdf.iterrows():
    embedding_str = "[" + ",".join(str(float(x)) for x in row["embedding"]) + "]"
    cursor.execute(insert_sql, (
        row["destination"], row["title"], row["summary"], embedding_str
    ))

conn.commit()
print(f"✅ Inserted {len(pdf)} embeddings into Lakebase")
cursor.close()
conn.close()

In [0]:
conn = psycopg2.connect(
    host=parsed.hostname, port=parsed.port or 5432, dbname=parsed.path.lstrip("/"),
    user=parsed.username, password=parsed.password, sslmode="require",
)
cursor = conn.cursor()

query_text = "ancient temples and historic culture"
query_vector = model.encode([query_text])[0]
query_str = "[" + ",".join(str(float(x)) for x in query_vector) + "]"

cursor.execute("""
    SELECT destination, title, embedding <=> %s::vector AS distance
    FROM destination_description_embeddings
    ORDER BY distance
    LIMIT 5;
""", (query_str,))

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

In [0]:
conn = psycopg2.connect(
    host=parsed.hostname, port=parsed.port or 5432, dbname=parsed.path.lstrip("/"),
    user=parsed.username, password=parsed.password, sslmode="require",
)
cursor = conn.cursor()

cursor.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

for row in cursor.fetchall():
    print(row[0])

cursor.close()
conn.close()